In [2]:
%pip install requests

  Using cached charset_normalizer-3.4.6-cp314-cp314-macosx_10_15_universal2.whl.metadata (40 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
Using cached charset_normalizer-3.4.6-cp314-cp314-macosx_10_15_universal2.whl (294 kB)
Using cached idna-3.11-py3-none-any.whl (71 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
Using cached certifi-2026.2.25-py3-none-any.whl (153 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [requests]1/5 [idna]

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Fetch the OpenAPI spec

In [3]:
import requests
import json

OPENAPI_URL = "https://api.apidash.dev/openapi.json"
BASE_URL = "https://api.apidash.dev"

def fetch_openapi(url):
    try:
        res = requests.get(url)
        res.raise_for_status()
        return res.json()
    except Exception as e:
        print("Error fetching OpenAPI:", e)
        return None

data = fetch_openapi(OPENAPI_URL)
print(json.dumps(data, indent=2))

{
  "openapi": "3.1.0",
  "info": {
    "title": "API Dash APIs",
    "description": "Open Source APIs for all!",
    "contact": {
      "name": "Know more about API Dash",
      "url": "https://apidash.dev/"
    },
    "version": "0.0.3"
  },
  "paths": {
    "/country/codes": {
      "get": {
        "tags": [
          "Country Data"
        ],
        "summary": "Get Country Code Dictionary",
        "operationId": "get_country_code_dictionary_country_codes_get",
        "responses": {
          "200": {
            "description": "Successful Response",
            "content": {
              "application/json": {
                "schema": {}
              }
            }
          }
        }
      }
    },
    "/country/data": {
      "get": {
        "tags": [
          "Country Data"
        ],
        "summary": "Get Country Data",
        "operationId": "get_country_data_country_data_get",
        "parameters": [
          {
            "name": "code",
            "in": "query

## Extract endpoints

In [4]:
def extract_endpoints(data):
    paths = data.get("paths", {})
    endpoints = []
    for path, methods in paths.items():
        for method, details in methods.items():
            # skip non-method keys like "parameters"
            if method not in ["get", "post", "put", "patch", "delete", "head", "options"]:
                continue
            endpoints.append({
                "path": path,
                "method": method.upper(),
                "summary": details.get("summary", ""),
                "tags": details.get("tags", []),
                "parameters": details.get("parameters", []),
                "requestBody": details.get("requestBody", None)
            })
    return endpoints

endpoints = extract_endpoints(data)
print(f"Found {len(endpoints)} endpoints\n")
for ep in endpoints[:5]:
    print(f"{ep['method']} {ep['path']} — {ep['summary']}")
    

Found 52 endpoints

GET /country/codes — Get Country Code Dictionary
GET /country/data — Get Country Data
GET /country/flag — Get Country Flag
GET /country/name — Get Country Name
GET /country/officialname — Get Official Country Name


## Rule-based auto-tagger

In [ ]:
# Predefined categories (aligned with API Dash's category system[i have checked that :-)])
CATEGORY_RULES = {
    "AI": ["ai", "openai", "anthropic", "gemini", "llm", "generative", "chat", "completion", "embedding"],
    "Finance": ["finance", "payment", "bank", "currency", "stock", "crypto", "invoice", "billing"],
    "Weather": ["weather", "forecast", "climate", "temperature", "humidity"],
    "Social Media": ["social", "twitter", "instagram", "facebook", "post", "feed", "profile"],
    "Developer Tools": ["convert", "parse", "format", "slug", "case", "encode", "decode", "humanize"],
    "Country & Geography": ["country", "flag", "subdivision", "region", "location", "geo"],
    "Authentication": ["auth", "login", "logout", "token", "oauth", "password"],
    "User Management": ["user", "profile", "account", "register", "signup"],
    "I/O & Files": ["file", "upload", "image", "form", "delay", "io"],
}

def infer_category(api_name, tags, paths):
    # Combine all signals into one searchable string
    signal = " ".join([
        api_name.lower(),
        " ".join(tags).lower(),
        " ".join(paths).lower()
    ])
    
    for category, keywords in CATEGORY_RULES.items():
        for keyword in keywords:
            if keyword in signal:
                return category
    
    return "General"

#tedt it
info = data.get("info", {})
all_tags = list(set(tag for ep in endpoints for tag in ep["tags"]))
all_paths = [ep["path"] for ep in endpoints]

category = infer_category(info.get("title", ""), all_tags, all_paths)
print(f"API Name: {info.get('title')}")
print(f"Tags found: {all_tags}")
print(f"Inferred Category: {category}")

API Name: API Dash APIs
Tags found: ['User Authentication', 'Country Data', 'Humanize', 'Text Conversion', 'Case Conversion', 'I/O', 'User Data']
Inferred Category: AI


## Extract Endpoints Properly(again)

In [6]:
def extract_endpoints(data):
    paths = data.get("paths", {})
    endpoints = []

    for path, methods in paths.items():
        for method, details in methods.items():
            endpoints.append({
                "path": path,
                "method": method.upper(),
                "tags": details.get("tags", [])
            })

    return endpoints

endpoints = extract_endpoints(data)

print("Total endpoints:", len(endpoints))
print(endpoints[:5])

Total endpoints: 52
[{'path': '/country/codes', 'method': 'GET', 'tags': ['Country Data']}, {'path': '/country/data', 'method': 'GET', 'tags': ['Country Data']}, {'path': '/country/flag', 'method': 'GET', 'tags': ['Country Data']}, {'path': '/country/name', 'method': 'GET', 'tags': ['Country Data']}, {'path': '/country/officialname', 'method': 'GET', 'tags': ['Country Data']}]


## Fix Our Category Test

In [7]:
info = data.get("info", {})

all_tags = list(set(tag for ep in endpoints for tag in ep["tags"]))
all_paths = [ep["path"] for ep in endpoints]

category = infer_category(info.get("title", ""), all_tags, all_paths)

print(f"API Name: {info.get('title')}")
print(f"Tags found: {all_tags}")
print(f"Inferred Category: {category}")

API Name: API Dash APIs
Tags found: ['User Authentication', 'Country Data', 'Humanize', 'Text Conversion', 'Case Conversion', 'I/O', 'User Data']
Inferred Category: AI


## Build ApiTemplate Structure

In [8]:
def build_template(data, endpoints):
    info = data.get("info", {})

    template = {
        "info": {
            "name": info.get("title", "Unknown API"),
            "description": info.get("description", "")
        },
        "category": None,
        "requests": []
    }

    # Infer category
    all_tags = list(set(tag for ep in endpoints for tag in ep["tags"]))
    all_paths = [ep["path"] for ep in endpoints]

    template["category"] = infer_category(
        info.get("title", ""),
        all_tags,
        all_paths
    )

    # Build request objects
    for ep in endpoints[:5]:  # limit for PoC
        template["requests"].append({
            "method": ep["method"],
            "url": BASE_URL + ep["path"],
            "headers": {},
            "body": {}
        })

    return template

template = build_template(data, endpoints)

print(json.dumps(template, indent=2))

{
  "info": {
    "name": "API Dash APIs",
    "description": "Open Source APIs for all!"
  },
  "category": "AI",
  "requests": [
    {
      "method": "GET",
      "url": "https://api.apidash.dev/country/codes",
      "headers": {},
      "body": {}
    },
    {
      "method": "GET",
      "url": "https://api.apidash.dev/country/data",
      "headers": {},
      "body": {}
    },
    {
      "method": "GET",
      "url": "https://api.apidash.dev/country/flag",
      "headers": {},
      "body": {}
    },
    {
      "method": "GET",
      "url": "https://api.apidash.dev/country/name",
      "headers": {},
      "body": {}
    },
    {
      "method": "GET",
      "url": "https://api.apidash.dev/country/officialname",
      "headers": {},
      "body": {}
    }
  ]
}


## Save now

In [9]:
def save_template(template, filename="sample_output.json"):
    with open(filename, "w") as f:
        json.dump(template, f, indent=2)

save_template(template)

print("Saved to sample_output.json")

Saved to sample_output.json


## Add Response Parsing

In [10]:
def extract_responses(data, endpoints):
    paths = data.get("paths", {})
    responses_map = {}

    for ep in endpoints:
        path = ep["path"]
        method = ep["method"].lower()

        details = paths.get(path, {}).get(method, {})
        responses = details.get("responses", {})

        responses_map[(path, method)] = list(responses.keys())

    return responses_map

responses_map = extract_responses(data, endpoints)

print(list(responses_map.items())[:5])

[(('/country/codes', 'get'), ['200']), (('/country/data', 'get'), ['200', '422']), (('/country/flag', 'get'), ['200', '422']), (('/country/name', 'get'), ['200', '422']), (('/country/officialname', 'get'), ['200', '422'])]
